# Human-in-the-Loop Approval for Irreversible Agent Actions

Anthropic's [autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy) — ~1M production tool calls — found that **0.8% of agent actions are irreversible** (moving money, deleting data, external communication) and that these should *"require mandatory human approval before execution."*

This recipe shows the pattern end to end with the Messages API:

1. Give Claude a **gate tool** it must call before any irreversible action
2. **Pause** execution and request a human approval bound to the *exact* action (not a generic 'OK')
3. Resume only on approval — and keep a **verifiable record** of who said yes

The teaching code simulates the approver so the notebook runs with just an `ANTHROPIC_API_KEY`. The *Production* section at the end covers real device-bound approvals (WebAuthn / passkeys) and offline-verifiable receipts.


In [ ]:
# %pip install anthropic
import json, os, hashlib
import anthropic

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY
MODEL = "claude-fable-5"


## 1. The gate tool

Claude gets two tools: the irreversible action (`release_payment`) and the gate (`require_human_approval`). The trick is in the descriptions — the irreversible tool's description *requires* the gate to have returned `proceed=true` first, so the model routes itself through the approval.


In [ ]:
GATE_TOOL = {
    "name": "require_human_approval",
    "description": (
        "REQUIRED before any irreversible action (payments, deletions, "
        "external sends). Returns proceed=true only after a human approves "
        "this EXACT action. Never execute the action unless proceed=true."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "action": {"type": "string", "description": "what will happen, specifically"},
            "amount": {"type": "number"},
            "target": {"type": "string", "description": "who/what is affected"},
        },
        "required": ["action", "target"],
    },
}

RELEASE_PAYMENT_TOOL = {
    "name": "release_payment",
    "description": (
        "Releases a payment (IRREVERSIBLE). You MUST call require_human_approval "
        "first and may only call this if it returned proceed=true."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"wire_id": {"type": "string"}, "amount": {"type": "number"}},
        "required": ["wire_id", "amount"],
    },
}


## 2. The approval — bound to the exact action

A real approval must answer *"approve **what**?"* cryptographically. We hash the canonical action context; the human approves **that hash**. Change one digit of the amount afterward and the approval no longer matches — this is what separates an audit-grade approval from a Slack 👍.

For the notebook, a `SIMULATED_APPROVER` plays the human. In production this is a push to a phone and a biometric signature (see the last section).


In [ ]:
SIMULATED_APPROVER_APPROVES = True   # flip to False to watch the agent get refused

def action_hash(args: dict) -> str:
    """Canonical hash of the exact action being approved."""
    canonical = json.dumps(args, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()

def request_human_approval(args: dict) -> dict:
    h = action_hash(args)
    print(f"⏸  HOLDING irreversible action for approval\n   action: {args}\n   binds to hash: {h[:16]}…")
    approved = SIMULATED_APPROVER_APPROVES  # ← in production: device signature here
    print(f"   human decision: {'✅ approved' if approved else '❌ rejected'}")
    return {"proceed": approved, "approved_action_hash": h if approved else None,
            "reason": "approved by named human" if approved else "rejected by named human"}


## 3. The agent loop


In [ ]:
def run_agent(prompt: str, max_turns: int = 6) -> str:
    messages = [{"role": "user", "content": prompt}]
    for _ in range(max_turns):
        resp = client.messages.create(model=MODEL, max_tokens=1024,
                                      tools=[GATE_TOOL, RELEASE_PAYMENT_TOOL], messages=messages)
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if block.type != "tool_use":
                continue
            if block.name == "require_human_approval":
                out = request_human_approval(block.input)
            elif block.name == "release_payment":
                out = {"released": True, "wire_id": block.input["wire_id"]}  # your real payout call
                print(f"💸 EXECUTED: release_payment {block.input}")
            else:
                out = {"error": f"unknown tool {block.name}"}
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})
    return "max turns reached"


In [ ]:
print(run_agent("Release wire/8841 for $82,000 to Vendor 8841. Follow the required approval process."))


Try `SIMULATED_APPROVER_APPROVES = False` and re-run: the agent is told the human rejected it, and `release_payment` never fires.

## What this proves — and what it doesn't

The simulated approver demonstrates the *control flow*, but a `True` in a notebook proves nothing about **who** approved. Production needs three properties the simulation lacks:

1. **The approval is a signature from a key the agent can never hold** — a device-bound WebAuthn/passkey credential (Face ID / Touch ID), so a compromised agent or operator cannot manufacture an approval.
2. **The signature binds the exact action** — challenge = SHA-256 of the canonical action context (as above), so approving \$82,000 is not approving \$820,000.
3. **Anyone can verify it later, offline** — pure public-key math, no trust in the operator's logs.

## Production references

- [EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol) (Apache-2.0) implements exactly this: device-bound approver signatures over canonicalized action context, separation of duties (approver ≠ initiator), one-time consumption, and offline verification (`npx @emilia-protocol/verify`). It ships as a [Claude Code plugin](https://github.com/emiliaprotocol/emilia-protocol/tree/main/integrations/claude-code-plugin) and an [Agent SDK PreToolUse hook](https://github.com/emiliaprotocol/emilia-protocol/tree/main/integrations/claude-agent-sdk), and the receipt format is an individual IETF Internet-Draft ([draft-schrock-ep-authorization-receipts](https://datatracker.ietf.org/doc/draft-schrock-ep-authorization-receipts/)).
- Vendor-neutral building blocks: [@simplewebauthn](https://simplewebauthn.dev/) (JS) and [python-fido2](https://github.com/Yubico/python-fido2) for device-bound signatures.
- The threshold for *which* actions need this maps to the \"earned trust\" curve in [Anthropic's autonomy research](https://www.anthropic.com/news/measuring-agent-autonomy): start strict, relax per-action as a clean approval history accrues.
